# Master Notebook — Model B: Swin-V2-Tiny
> Comprehensive execution environment for **Google Colab** and **Kaggle**.
Handles Drive mounting, Kaggle dataset downloading, and automated directory management.

### Step 1: Environment Setup & Dependencies

In [1]:
!pip install -q albumentations==1.4.1
!pip install -q timm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.5/130.5 kB 4.5 MB/s eta 0:00:00


### Step 2: Drive Mounting & Directory Management (Colab Only)

In [2]:
import os
import sys
import zipfile

IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_COLAB = not IS_KAGGLE and ("COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules)

if IS_COLAB:
    from google.colab import drive
    from google.colab import files
    # Mount Drive
    drive.mount('/content/drive', force_remount=True)

    # Base Directories
    DRIVE_BASE = "/content/drive/MyDrive/DR_Thesis"
    DATA_DIR = f"{DRIVE_BASE}/datasets/aptos2019-blindness-detection"
    CODE_DIR = f"{DRIVE_BASE}/model_B_swin"

    os.makedirs(DATA_DIR, exist_ok=True)
    os.makedirs(CODE_DIR, exist_ok=True)

    # Sync code to local Colab instance for faster execution
    print("Syncing code from Drive...")
    !rsync -av {CODE_DIR}/ ./

    # Dataset Download Logic
    if not os.path.exists(f"{DATA_DIR}/train.csv"):
        print("\nAPTOS dataset not found in Drive. Preparing to download via Kaggle API.")

        if not os.path.exists("/root/.kaggle/access_token") and "KAGGLE_API_TOKEN" not in os.environ:
            import getpass
            print("To download the dataset, please enter your Kaggle API Token (starts with KGAT_).")
            print("You can find this at Kaggle -> Settings -> API -> Create New Token")
            kaggle_token = getpass.getpass("Kaggle API Token: ")

            os.makedirs("/root/.kaggle", exist_ok=True)
            with open("/root/.kaggle/access_token", "w") as f:
                f.write(kaggle_token.strip())
            os.chmod("/root/.kaggle/access_token", 0o600)
            print("Token saved successfully!")

        print("Downloading APTOS dataset...")
        !kaggle competitions download -c aptos2019-blindness-detection -p {DATA_DIR}

        print("Extracting dataset...")
        zip_path = f"{DATA_DIR}/aptos2019-blindness-detection.zip"
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(DATA_DIR)

        print("Cleaning up zip file...")
        os.remove(zip_path)
        print("Dataset ready!")
    else:
        print("\nAPTOS dataset already exists in Drive.")

elif IS_KAGGLE:
    print("Running on Kaggle. Syncing code dataset...")
    import glob
    code_dirs = glob.glob("/kaggle/input/**/*code*/", recursive=True)
    if code_dirs:
        !rsync -av {code_dirs[0]} ./
    else:
        print("Warning: Code dataset not found in /kaggle/input/")
else:
    print("Running locally.")

Running on Kaggle. Syncing code dataset...
sending incremental file list
./
config.py
dataset.py
evaluate.py
losses.py
metrics.py
model.py
preprocessing.py
train.py

sent 67,419 bytes  received 171 bytes  135,180.00 bytes/sec
total size is 66,824  speedup is 0.99


### Step 3: Execute Training Pipeline
The pipeline will auto-detect Colab/Kaggle and route paths accordingly.

In [3]:
!python train.py

  [Kaggle] APTOS=/kaggle/input/competitions/aptos2019-blindness-detection
  [Kaggle] Output=/kaggle/working/outputs
Device: cuda
GPU: Tesla T4

  ModelB_SwinV2T — Fine-tuning on APTOS (ImageNet → APTOS)
  Scanning APTOS directory for images...
  APTOS: 3662 images loaded
  Class distribution:
diagnosis
0    1805
1     370
2     999
3     193
4     295
Name: count, dtype: int64

  Fold 1: train=2929, val=733
  Fold 2: train=2929, val=733
  Fold 3: train=2930, val=732
  Fold 4: train=2930, val=732
  Fold 5: train=2930, val=732

  ┌─ Fold 1/5

  ModelB_SwinV2T | Fold 1/5 | LR=5.0e-06 | Epochs=30
Downloading: "https://download.pytorch.org/models/swin_v2_t-b137f0e2.pth" to /root/.cache/torch/hub/checkpoints/swin_v2_t-b137f0e2.pth
100%|█████████████████████████████████████████| 109M/109M [00:00<00:00, 201MB/s]
  Gradient checkpointing enabled for Swin-V2-T
  ⟳ Resuming from checkpoint: /kaggle/input/datasets/ayushuegjng/dr-model-b-checkpoints/ModelB_SwinV2T_fold0_stage2_state.pth
  Resumed a

### Step 4: Evaluate and Export for Ensemble

In [4]:
!python evaluate.py

  [Kaggle] APTOS=/kaggle/input/competitions/aptos2019-blindness-detection
  [Kaggle] Output=/kaggle/working/outputs

############################################################
  EVALUATION: ModelB_SwinV2T
############################################################
  Scanning APTOS directory for images...
  APTOS: 3662 images loaded
  Class distribution:
diagnosis
0    1805
1     370
2     999
3     193
4     295
Name: count, dtype: int64

  Fold 1: train=2929, val=733
  Fold 2: train=2929, val=733
  Fold 3: train=2930, val=732
  Fold 4: train=2930, val=732
  Fold 5: train=2930, val=732

  Evaluating ModelB_SwinV2T — Fold 1...
  QWK=0.9144 | Acc=0.8240
  Saved: /kaggle/working/outputs/plots/ModelB_SwinV2T_fold1_cm.pdf
  Saved: /kaggle/working/outputs/plots/ModelB_SwinV2T_fold1_roc.pdf
  Saved: /kaggle/working/outputs/plots/ModelB_SwinV2T_fold1_pr.pdf
2026-05-01 09:19:13.742346: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting